In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate, cross_val_predict
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

#1. 某些column的值是string，用one hot encoding處理
df = pd.read_csv("../Data/diabetes_prediction_dataset.csv")
df = pd.get_dummies(df, columns=["smoking_history"], dtype=int)
df = pd.get_dummies(df, columns=["gender"], dtype=int)

#2. 將diabetes設定為y(要預測的)，其餘欄位設定為x(輸入)
x = df.drop("diabetes", axis=1)
y = df["diabetes"]

#3. 建立4-fold validation
'''
假設你的資料比例是：
diabetes = 0 → 91.5%
diabetes = 1 → 8.5%

StratifiedKFold 會盡量讓每一 Fold 都保持類似比例：
Fold 1 → 約 91.5% : 8.5%
Fold 2 → 約 91.5% : 8.5%
Fold 3 → 約 91.5% : 8.5%
Fold 4 → 約 91.5% : 8.5%
'''
skf = StratifiedKFold(
    n_splits=4,  #分為4 fold
    shuffle=True,  #分成4 fold之前，先把資料順序打亂
    random_state=17  #既然是資料打亂，之後輸入這個random_state=17，打亂的方式都一樣了
)

'''
#4. 建立Decision Tree並執行model
for depth in range(1,7):
    model = DecisionTreeClassifier(
        max_depth=depth,
        min_samples_leaf=10,
        random_state=1
    )

    scores = cross_validate(
        model,
        x,
        y,
        cv=skf,
        scoring={  #左邊是名稱，右邊是sklearn真正使用的評分方式
            "accuracy": "accuracy",
            "precision": "precision",
            "recall": "recall",
            "f1": "f1",
        }
    )

    print(
        f"depth={depth:2d} | "
        f"accuracy={scores['test_accuracy'].mean():.4f} | "
        f"precision={scores['test_precision'].mean():.4f} | "
        f"recall={scores['test_recall'].mean():.4f} | "
        f"f1={scores['test_f1'].mean():.4f} | "
    )
    '''

model = DecisionTreeClassifier(
    max_depth=2,
    min_samples_leaf=10,
    random_state=1
)

# 取得 diabetes=1 的預測機率
y_prob = cross_val_predict(
    model,
    x,
    y,
    cv=skf,
    method="predict_proba"
)[:, 1]

# for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
for threshold in [0.02964, 0.02965, 0.02968, 0.03034, 0.031, 0.5, 0.8]:
    y_pred = (y_prob >= threshold).astype(int)

    print(
        f"threshold={threshold:.5f} | "
        f"accuracy={accuracy_score(y, y_pred):.5f} | "
        f"precision={precision_score(y, y_pred):.5f} | "
        f"recall={recall_score(y, y_pred):.5f} | "
        f"f1={f1_score(y, y_pred):.5f}"
    )

values, counts = np.unique(y_prob, return_counts=True)

for value, count in zip(values, counts):
    print(f"機率={value:.5f}, 數量={count}")




threshold=0.02964 | accuracy=0.30658 | precision=0.10186 | recall=0.91565 | f1=0.18333
threshold=0.02965 | accuracy=0.30658 | precision=0.10186 | recall=0.91565 | f1=0.18333
threshold=0.02968 | accuracy=0.74978 | precision=0.21736 | recall=0.74741 | f1=0.33678
threshold=0.03034 | accuracy=0.97187 | precision=1.00000 | recall=0.66906 | f1=0.80172
threshold=0.03100 | accuracy=0.97187 | precision=1.00000 | recall=0.66906 | f1=0.80172
threshold=0.50000 | accuracy=0.97187 | precision=1.00000 | recall=0.66906 | f1=0.80172
threshold=0.80000 | accuracy=0.97187 | precision=1.00000 | recall=0.66906 | f1=0.80172
機率=0.02964, 數量=23592
機率=0.02965, 數量=23591
機率=0.02968, 數量=23589
機率=0.03034, 數量=23541
機率=1.00000, 數量=5687


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

df = pd.read_csv("../Data/diabetes_prediction_dataset.csv")
df = pd.get_dummies(df, columns=["smoking_history"], dtype=int)
df = pd.get_dummies(df, columns=["gender"], dtype=int)

x = df.drop("diabetes", axis=1)
y = df["diabetes"]

skf = StratifiedKFold(
    n_splits=4, 
    shuffle=True,  
    random_state=17  
)

# 建立 Logistic Regression model
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=100)
)

# 取得 diabetes=1 的預測機率
y_prob = cross_val_predict(
    model,
    x,
    y,
    cv=skf,
    method="predict_proba"
)[:, 1]

# for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
for threshold in [0.02964, 0.02965, 0.02968, 0.03034, 0.031, 0.5, 0.8]:
    y_pred = (y_prob >= threshold).astype(int)

    print(
        f"threshold={threshold:.5f} | "
        f"accuracy={accuracy_score(y, y_pred):.5f} | "
        f"precision={precision_score(y, y_pred):.5f} | "
        f"recall={recall_score(y, y_pred):.5f} | "
        f"f1={f1_score(y, y_pred):.5f}"
    )

rounded_prob = np.round(y_prob, 4)
values, counts = np.unique(rounded_prob, return_counts=True)

for value, count in zip(values, counts):
    print(f"機率={value:.4f}, 數量={count}")


threshold=0.02964 | accuracy=0.79440 | precision=0.28697 | recall=0.95565 | f1=0.44140
threshold=0.02965 | accuracy=0.79441 | precision=0.28698 | recall=0.95565 | f1=0.44141
threshold=0.02968 | accuracy=0.79455 | precision=0.28712 | recall=0.95565 | f1=0.44158
threshold=0.03034 | accuracy=0.79683 | precision=0.28929 | recall=0.95435 | f1=0.44399
threshold=0.03100 | accuracy=0.79911 | precision=0.29156 | recall=0.95353 | f1=0.44657
threshold=0.50000 | accuracy=0.96016 | precision=0.86620 | recall=0.62835 | f1=0.72835
threshold=0.80000 | accuracy=0.95519 | precision=0.98109 | recall=0.48212 | f1=0.64653
機率=0.0000, 數量=9764
機率=0.0001, 數量=6723
機率=0.0002, 數量=3885
機率=0.0003, 數量=2775
機率=0.0004, 數量=2240
機率=0.0005, 數量=1841
機率=0.0006, 數量=1596
機率=0.0007, 數量=1395
機率=0.0008, 數量=1236
機率=0.0009, 數量=1121
機率=0.0010, 數量=919
機率=0.0011, 數量=952
機率=0.0012, 數量=870
機率=0.0013, 數量=863
機率=0.0014, 數量=742
機率=0.0015, 數量=736
機率=0.0016, 數量=703
機率=0.0017, 數量=643
機率=0.0018, 數量=627
機率=0.0019, 數量=569
機率=0.0020, 數量=543
機率=